# Codificación de variables categóricas

Este notebook explica las técnicas más comunes para codificar variables categóricas con un ejemplo más largo y práctico.

## Objetivos

- Diferenciar variables nominales y ordinales.
- Aplicar `OneHotEncoder`, `OrdinalEncoder` y una frecuencia simple.
- Comparar cuándo usar cada técnica.
- Integrar la codificación en un pipeline.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

rng = np.random.default_rng(123)
pd.set_option('display.max_columns', None)

## 1. Crear dataset

In [2]:
n = 1400
df = pd.DataFrame({
    'edad': rng.integers(18, 70, size=n),
    'ingreso': rng.normal(3200, 1000, size=n).clip(600, 12000),
    'ciudad': rng.choice(['Bogotá', 'Medellín', 'Cali', 'Barranquilla', 'Cartagena'], size=n),
    'producto': rng.choice(['Tarjeta', 'Consumo', 'Libre Inversión', 'Vehículo'], size=n, p=[0.35, 0.30, 0.20, 0.15]),
    'canal': rng.choice(['App', 'Web', 'Call Center', 'Sucursal'], size=n, p=[0.45, 0.25, 0.10, 0.20]),
    'educacion': rng.choice(['Bachiller', 'Técnico', 'Profesional', 'Posgrado'], size=n, p=[0.25, 0.25, 0.35, 0.15]),
    'riesgo_textual': rng.choice(['Bajo', 'Medio', 'Alto'], size=n, p=[0.45, 0.35, 0.20])
})

risk_num = df['riesgo_textual'].map({'Bajo': -0.7, 'Medio': 0.1, 'Alto': 0.9})
score = (
    0.015 * df['edad'] - 0.0003 * df['ingreso'] + risk_num
    + df['producto'].map({'Tarjeta': 0.2, 'Consumo': 0.25, 'Libre Inversión': 0.35, 'Vehículo': -0.1})
    + df['canal'].map({'App': 0.1, 'Web': 0.0, 'Call Center': 0.2, 'Sucursal': -0.1})
)
p = 1 / (1 + np.exp(-(score - score.mean())))
df['target'] = rng.binomial(1, p)
df.head()

,edad,ingreso,ciudad,producto,canal,educacion,riesgo_textual,target
0,18,3712.194922,Medellín,Tarjeta,App,Profesional,Bajo,0
1,53,4687.852122,Barranquilla,Vehículo,Sucursal,Bachiller,Alto,1
2,48,2339.232277,Bogotá,Tarjeta,Web,Profesional,Medio,1
3,20,3136.207191,Bogotá,Libre Inversión,Call Center,Técnico,Bajo,0
4,65,5649.968411,Bogotá,Consumo,Sucursal,Profesional,Alto,1


## 2. Introducir algunos faltantes categóricos

In [3]:
for col, prob in [('ciudad', 0.04), ('canal', 0.03), ('educacion', 0.05)]:
    mask = rng.random(n) < prob
    df.loc[mask, col] = np.nan

df.isna().mean()

edad              0.000000
ingreso           0.000000
ciudad            0.042857
producto          0.000000
canal             0.022857
educacion         0.044286
riesgo_textual    0.000000
target            0.000000
dtype: float64

## 3. ¿Qué tipo de categóricas tenemos?

- **Nominales**: ciudad, producto, canal. No tienen orden natural.
- **Ordinales**: riesgo_textual, educación. Sí podemos imponer orden.

In [4]:
X = df.drop(columns='target')
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=123)

num_cols = ['edad', 'ingreso']
nominal_cols = ['ciudad', 'producto', 'canal']
ordinal_cols = ['riesgo_textual', 'educacion']

## 4. One-Hot Encoding para nominales

In [5]:
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
demo = X_train[nominal_cols].head(8)
demo_encoded = ohe.fit_transform(demo)
demo_ohe_df = pd.DataFrame(demo_encoded, columns=ohe.get_feature_names_out(nominal_cols), index=demo.index)
pd.concat([demo, demo_ohe_df], axis=1)

,ciudad,producto,canal,ciudad_Bogotá,ciudad_Cali,ciudad_Cartagena,ciudad_nan,producto_Consumo,producto_Libre Inversión,producto_Tarjeta,producto_Vehículo,canal_App,canal_Call Center,canal_Sucursal,canal_Web
552,Cali,Vehículo,Sucursal,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
762,Cartagena,Tarjeta,App,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
585,Bogotá,Consumo,Call Center,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1066,NaN,Vehículo,Sucursal,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
1275,Cartagena,Consumo,Call Center,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
297,NaN,Tarjeta,Web,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
1334,Cartagena,Libre Inversión,Web,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
1358,Cali,Tarjeta,Web,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0


## 5. Ordinal Encoding para variables con orden

In [6]:
ord_encoder = OrdinalEncoder(categories=[['Bajo', 'Medio', 'Alto'], ['Bachiller', 'Técnico', 'Profesional', 'Posgrado']])
ord_demo = X_train[ordinal_cols].head(8).copy()
ord_demo_imputed = ord_demo.fillna({'educacion': 'Bachiller'})
ord_encoded = ord_encoder.fit_transform(ord_demo_imputed)
ord_df = pd.DataFrame(ord_encoded, columns=[c + '_cod' for c in ordinal_cols], index=ord_demo.index)
pd.concat([ord_demo, ord_df], axis=1)

,riesgo_textual,educacion,riesgo_textual_cod,educacion_cod
552,Bajo,Profesional,0.0,2.0
762,Alto,Bachiller,2.0,0.0
585,Bajo,Profesional,0.0,2.0
1066,Medio,Técnico,1.0,1.0
1275,Medio,Bachiller,1.0,0.0
297,Bajo,Técnico,0.0,1.0
1334,Alto,Profesional,2.0,2.0
1358,Medio,Posgrado,1.0,3.0


## 6. Frecuencia / target-agnostic count encoding simple

No vamos a usar una librería externa; haremos una versión sencilla basada en frecuencia relativa.

In [7]:
freq_map = X_train['ciudad'].value_counts(normalize=True, dropna=False)
X_train[['ciudad']].assign(ciudad_freq=X_train['ciudad'].map(freq_map)).head(10)

,ciudad,ciudad_freq
552,Cali,0.207619
762,Cartagena,0.171429
585,Bogotá,0.206667
1066,NaN,0.039048
1275,Cartagena,0.171429
297,NaN,0.039048
1334,Cartagena,0.171429
1358,Cali,0.207619
1274,Medellín,0.182857
0,Medellín,0.182857


## 7. Pipeline completo: numéricas + nominales + ordinales

In [8]:
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

nom_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore'))
])

ord_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ord', OrdinalEncoder(categories=[['Bajo', 'Medio', 'Alto'], ['Bachiller', 'Técnico', 'Profesional', 'Posgrado']]))
])

preprocessor = ColumnTransformer([
    ('num', num_pipe, num_cols),
    ('nom', nom_pipe, nominal_cols),
    ('ord', ord_pipe, ordinal_cols)
])

model = Pipeline([
    ('prep', preprocessor),
    ('clf', LogisticRegression(max_iter=2000))
])

model.fit(X_train, y_train)
pred = model.predict_proba(X_test)[:, 1]
print('AUC:', round(roc_auc_score(y_test, pred), 4))

AUC: 0.6997


## 8. Inspeccionar dimensionalidad final

In [9]:
Xt = preprocessor.fit_transform(X_train)
print('Filas:', Xt.shape[0])
print('Columnas después de codificar:', Xt.shape[1])

Filas: 1050
Columnas después de codificar: 17


## 9. ¿Qué pasa si tratamos todo como ordinal?

Esto suele ser una mala idea para nominales, pero sirve como contraste.

In [10]:
bad_preprocessor = ColumnTransformer([
    ('num', num_pipe, num_cols),
    ('all_cat_bad', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ord', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
    ]), nominal_cols + ordinal_cols)
])

bad_model = Pipeline([
    ('prep', bad_preprocessor),
    ('clf', LogisticRegression(max_iter=2000))
])
bad_model.fit(X_train, y_train)
bad_pred = bad_model.predict_proba(X_test)[:, 1]
print('AUC codificación ordinal indiscriminada:', round(roc_auc_score(y_test, bad_pred), 4))

AUC codificación ordinal indiscriminada: 0.5809


## 10. Conclusiones

- One-hot es el baseline natural para nominales.
- OrdinalEncoder tiene sentido cuando sí existe un orden.
- Codificar mal una variable categórica puede inducir relaciones artificiales.
- Siempre conviene empaquetar la lógica en pipelines.